# U-Net building segmentation

Semantic segmentation of building footprints in aerial imagery. The model is a
U-Net with an ImageNet-pretrained ResNet-34 encoder (via
`segmentation_models_pytorch`), trained on the Massachusetts Buildings dataset
and then checked on Polish orthophotos of Wieliczka to see how far it travels.

Full runs: `train.py`, `predict.py`. This notebook shows the data and a short run.

In [ ]:
import os, sys
# Work from the project root, so imports and data paths match the scripts.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from data import DATA_ROOT, ClassMap, decode_mask, find_pairs, read_rgb

class_map = ClassMap.from_csv(DATA_ROOT / "label_class_dict.csv")
print(class_map.names, class_map.palette.tolist())

train_pairs, val_pairs, test_pairs = find_pairs(DATA_ROOT)
len(train_pairs), len(val_pairs), len(test_pairs)

One tile with its ground truth. Note the Massachusetts labels code buildings
in **red**, not white - worth checking before training.

In [ ]:
image_path, mask_path = train_pairs[0]
image = read_rgb(image_path)
truth = class_map.to_rgb(decode_mask(mask_path, class_map))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image)
axes[1].imshow(truth)
for ax in axes:
    ax.axis("off")

## Training

A short run to check the pipeline; the real model comes from `python train.py`
(12 epochs reproduces the original notebook settings, 100 epochs is the
converged model).

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "train.py", "--epochs", "2", "--out", "runs/notebook-demo"], check=True)

## Prediction on the held-out tiles

In [ ]:
import torch

from train import BuildingsDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.load("runs/notebook-demo/best_model.pth", map_location=device, weights_only=False)
model.eval()

shown = min(3, len(test_pairs))
dataset = BuildingsDataset(test_pairs[:shown], class_map)
fig, axes = plt.subplots(shown, 3, figsize=(12, 4 * shown), squeeze=False)
for i in range(shown):
    image, truth = dataset[i]
    with torch.no_grad():
        scores = model(torch.from_numpy(image[None]).to(device))
    predicted = scores.argmax(1).squeeze(0).cpu().numpy()
    for j, panel in enumerate((read_rgb(test_pairs[i][0]),
                               class_map.to_rgb(truth.argmax(0)),
                               class_map.to_rgb(predicted))):
        axes[i, j].imshow(panel)
        axes[i, j].axis("off")